In [1]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta,time
from scipy import stats
import json
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
from matplotlib.ticker import MultipleLocator


In [3]:
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler


In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent

    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [ ]:
userInputDataRaw = loadDataFromFile("UserPrevious experiments-preprocessed")
timeSeriesData_BIGraw = loadDataFromFile("Data:Previous experiments-preprocessed")

In [ ]:
userInputDataRaw

In [ ]:
timeSeriesData_BIGraw

In [ ]:
timeSeriesData_BIGraw = timeSeriesData_BIGraw.set_index("seconds",drop=False)

In [ ]:
timeSeriesData_BIGraw

In [ ]:
print(timeSeriesData_BIGraw.loc[timeSeriesData_BIGraw["keys"] == 157].max())
print(userInputDataRaw.loc[157])


In [ ]:
userInputDataRaw.index

In [ ]:
timeSeriesData_BIGraw["keys"].unique()

In [ ]:
print(timeSeriesData_BIGraw["keys"].unique().size)

In [ ]:
a = userInputDataRaw.index
b = timeSeriesData_BIGraw["keys"].unique()
diff_all = list(set(a).symmetric_difference(set(b)))
print(diff_all)  
userInputDataRaw.index = timeSeriesData_BIGraw["keys"].unique()
print(userInputDataRaw.index)

In [ ]:
userInputDataRaw.head(20)

In [ ]:
# Convert back to timedelta
timeSeriesData_BIGraw['timestamp'] = pd.to_timedelta(timeSeriesData_BIGraw['timestamp'], unit='ms')

# Convert back to datetime

timeSeriesData_BIGraw ["datetime_timestamp"]= timeSeriesData_BIGraw['datetime_timestamp'].transform(
    lambda x: pd.to_datetime(x, unit='ms')
)


columns_datetime= [
       'date of experiment', 'actual timestamp StartingExperiment',
       'actual timestamp EndingExperiment']
columns_timedelta = ['time taken total','time taken before insertion',
       'timestamp InsertingSource timedelta',
       'time taken after insertion']
# Ensure target columns are of object type before assignment
userInputDataRaw[columns_datetime] = userInputDataRaw[columns_datetime].astype('object')
userInputDataRaw[columns_timedelta] = userInputDataRaw[columns_timedelta].astype('object')

userInputDataRaw.loc[:,columns_datetime] = userInputDataRaw.loc[:,columns_datetime].apply(lambda x:pd.to_datetime(x, unit='ms'))
userInputDataRaw.loc[:,columns_timedelta] = userInputDataRaw.loc[:,columns_timedelta].apply(lambda x:pd.to_timedelta(x, unit='ms'))

In [ ]:
userInputDataRaw["room"].unique()

In [ ]:
timeSeriesData_BIG = timeSeriesData_BIGraw.copy()
userInputData = userInputDataRaw.copy()

In [ ]:
timeSeriesData_BIG

In [ ]:
userInputData.loc[157]

In [ ]:
userInputData["room"].unique()

In [ ]:
#keep the data from the last set experiments made that have the 3 sensors in a triangle shape,they have 16 particular points in the space and the door is closed
room_mask = userInputData["room"].isin(['Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0', 'Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 , Δ:2.00'])

open_door_mask = userInputData["are-doors-opened"] != "on"
mask = room_mask & open_door_mask
userInputData = userInputData.loc[mask]
#grab all the data that are contained in those experiments
timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]

In [ ]:
df = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"] == 157].max()
df

In [ ]:
timeSeriesData_BIG

In [ ]:
userInputData = userInputData.iloc[7:] 

In [ ]:
subset = timeSeriesData_BIG.loc[(timeSeriesData_BIG["keys"] == 157) & (timeSeriesData_BIG["sensors"] =="Id=2:BME680:breathVocEquivalent")]
subset_Show = subset.loc[20:80]
subset_Show.head(60)

In [ ]:
#if we want to keep

In [ ]:
#take standard scaler
if "VOC rolling average standard scaled per sensor" not in timeSeriesData_BIG.columns:
    timeSeriesData_BIG["VOC standard scaled per sensor"] = np.nan
if "VOC rolling average standard scaled per sensor gradient" not in timeSeriesData_BIG.columns:
    timeSeriesData_BIG["VOC standard scaled per sensor gradient"] = np.nan
    
keys = timeSeriesData_BIG["keys"].unique()
for key in keys:
    scaler = StandardScaler()
    sensors = timeSeriesData_BIG["sensors"].unique()
    for sensor in sensors:
        mask = (timeSeriesData_BIG["keys"] == key) & (timeSeriesData_BIG["sensors"]==sensor)
        timeSeriesData_BIG.loc[mask,"VOC rolling average standard scaled per sensor"] = scaler.fit_transform(timeSeriesData_BIG.loc[mask,"VOC rolling average-capped"].to_numpy().reshape(-1,1))
        timeSeriesData_BIG.loc[mask,"VOC rolling average standard scaled per sensor gradient"] = np.gradient(timeSeriesData_BIG.loc[mask,"VOC rolling average standard scaled per sensor"].to_numpy())


In [ ]:
timeSeriesData_BIG

In [ ]:
key_sensor_grouping = timeSeriesData_BIG.groupby(["keys","sensors"]).groups
for key_sensor_key in key_sensor_grouping.keys():
    key = key_sensor_key[0]
    sensor = key_sensor_key[1]
    one_part_of_mask = (timeSeriesData_BIG["keys"] == key ) & (timeSeriesData_BIG["sensors"]==sensor ) 
    column_to_insert = f"{sensor} gradient before mean"
    if column_to_insert not in userInputData.columns:
        userInputData[column_to_insert] = np.nan
    mask =  one_part_of_mask & (timeSeriesData_BIG["seconds"] <= 0 )  

    value = timeSeriesData_BIG.loc[mask,"VOC rolling average gradient"].mean()
    userInputData.at[key,column_to_insert] = value
    column_to_insert = f"{sensor} gradient after mean"
    if column_to_insert not in userInputData.columns:
        userInputData[column_to_insert] = np.nan
    mask =  one_part_of_mask & (timeSeriesData_BIG["seconds"] > 0 )  
    value = timeSeriesData_BIG.loc[mask,"VOC rolling average gradient"].mean()
    userInputData.at[key,column_to_insert] = value


    

In [ ]:
userInputData

In [ ]:
sensors = timeSeriesData_BIG["sensors"].unique()
for sensor in sensors:
    column_before_mean = f"{sensor} gradient before mean"
    column_after_mean = f"{sensor} gradient after mean"
    column_to_insert = f"{sensor} is gradient mean after biggen than before mean"
    if column_to_insert not in timeSeriesData_BIG.columns:
        userInputData[column_to_insert] = np.nan
    userInputData[column_to_insert] = userInputData[column_to_insert] = userInputData[column_after_mean] > userInputData[column_before_mean]

In [ ]:
userInputData

In [ ]:
userInputData

In [ ]:
a = np.sort(userInputData.loc[:, "Euclidian distance to Id=0:BME680:breathVocEquivalent"].unique())
a

In [ ]:
np.sort(userInputData.loc[:, "Euclidian distance to Id=1:BME680:breathVocEquivalent"].unique())

In [ ]:
np.sort(userInputData.loc[:, "Euclidian distance to Id=2:BME680:breathVocEquivalent"].unique())

In [ ]:
sensors = timeSeriesData_BIG["sensors"].unique()
euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
userInputData[euclidian_distances_columns] = userInputData[euclidian_distances_columns].round(2) 

In [ ]:
userInputData[euclidian_distances_columns]

In [ ]:
for column in euclidian_distances_columns:
    sorted_column = np.sort((userInputData[column].unique()))
    print(len(sorted_column))
    print(sorted_column)

In [ ]:
userInputData[["Id=0:BME680:breathVocEquivalent is gradient mean after biggen than before mean","Id=1:BME680:breathVocEquivalent is gradient mean after biggen than before mean","Id=2:BME680:breathVocEquivalent is gradient mean after biggen than before mean"]].sum()

In [ ]:
userInputData.loc[userInputData["Id=1:BME680:breathVocEquivalent is gradient mean after biggen than before mean"]==False]

In [ ]:
userInputData.loc[userInputData["Id=2:BME680:breathVocEquivalent is gradient mean after biggen than before mean"]==False]

In [ ]:
df = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"] == 157].max()
df

In [ ]:
#create gradient

In [ ]:
# Split back into dict
dict_of_timeseries = {k: v.drop(columns="keys").reset_index(drop=True) 
             for k, v in timeSeriesData_BIG.groupby("keys")}

In [ ]:
dict_of_timeseries[157].max()

In [ ]:
for index,data in dict_of_timeseries.items():
    dict_of_timeseries[index] = dict_of_timeseries[index].set_index("seconds",drop=False)

In [ ]:
all_positions = userInputData.loc[:, ["Euclidian distance to Id=0:BME680:breathVocEquivalent", "Euclidian distance to Id=1:BME680:breathVocEquivalent","Euclidian distance to Id=2:BME680:breathVocEquivalent"]]

In [ ]:
userInputData.loc[:, "Euclidian distance to Id=2:BME680:breathVocEquivalent"].unique()

In [ ]:
dict_of_timeseries[154]

In [ ]:
userInputData

In [ ]:
timeSeriesData_BIG

In [ ]:
def plot_position(userInputData,sample_row_of_the_group):
    plt.figure(figsize=(6, 6))  
    position_of_sensors = userInputData.iloc[-1]
    all_positions = userInputData.loc[:, ["x axis", "y axis"]]
    # Extra points
    extra_positions = np.array([
        [position_of_sensors["position of Id=0:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=0:BME680:breathVocEquivalent-y axis"]],
    
        [position_of_sensors["position of Id=1:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=1:BME680:breathVocEquivalent-y axis"]],
        [position_of_sensors["position of Id=2:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=2:BME680:breathVocEquivalent-y axis"]]
    ])
    extra_ids = ["id0","id1", "id2"]
    
    
    room_length = 4.0
    room_width = 3.25
    
    
    # Create scatterplot of the sources of all the particular setup
    #sns.scatterplot(x=positions[:,0], y=positions[:,1])
    sns.scatterplot(data=all_positions, x="x axis", y="y axis", color='blue', s=100, label='User Input Data')
    
    
    
    # Add the positions of sensors
    sns.scatterplot(x=extra_positions[:,0], y=extra_positions[:,1], color='red', s=100)
    x_sensor_highlight = sample_row_of_the_group["x axis"]
    y_sensor_highlight = sample_row_of_the_group["y axis"]
    # Plot a hollow circle around it
    plt.scatter(x_sensor_highlight, y_sensor_highlight, s=500, facecolors='none', edgecolors='green', linewidths=2, label='Highlighted point')  
    # Draw lines and annotate distances
    distances_from_sensors = (
        sample_row_of_the_group["Euclidian distance to Id=0:BME680:breathVocEquivalent"],
        sample_row_of_the_group["Euclidian distance to Id=1:BME680:breathVocEquivalent"],
        sample_row_of_the_group["Euclidian distance to Id=2:BME680:breathVocEquivalent"]
    )
    
    for i, (x, y) in enumerate(extra_positions):
        plt.plot([x_sensor_highlight, x], [y_sensor_highlight, y], color='red', linewidth=0.7, alpha=0.7)
        
        if distances_from_sensors is not None:
            # Midpoint of the line for annotation
            mid_x = (x_sensor_highlight + x) / 2
            mid_y = (y_sensor_highlight + y) / 2
            plt.text(mid_x, mid_y, f"{distances_from_sensors[i]:.2f}", color='red', fontsize=8, ha='center', va='center',
                         bbox=dict(facecolor='white', edgecolor='none', alpha=0.6, pad=1))
    # Annotate extra points with their IDs
    for i, (x, y) in enumerate(extra_positions):
        plt.text(x, y, extra_ids[i], fontsize=12, ha='right', va='bottom', color='red')
    
    
    # Set axis limits
    plt.xlim(-room_width, 0)
    plt.ylim(0, room_length)
    
    # Add grid
    plt.grid(True, which="both", linestyle="--", linewidth=0.7, alpha=0.7)
    # Smaller legend
    plt.legend(fontsize=8, markerscale=0.8, labelspacing=0.4)

    plt.show()

In [ ]:
def printDataBasedOnDate(column_to_print,userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping):
    
    column_names_keys_color_values = {"Id=0:BME680:breathVocEquivalent":"blue","Id=1:BME680:breathVocEquivalent":"green","Id=2:BME680:breathVocEquivalent":"yellow"}
    
    for group_name,indexes_of_the_group in room_other_grouping.items(): 
        timeSeriesData_BIG_copy = timeSeriesData_BIG.copy() 

        if ("position"  in type_of_other_grouping):
            sample_row_of_the_group = userInputData.loc[indexes_of_the_group[0],:]
            
            plot_position(userInputData,sample_row_of_the_group)       
        print(f"group_name {group_name}")
        print(f"indexes_of_the_group {indexes_of_the_group}")
        data = timeSeriesData_BIG_copy.loc[timeSeriesData_BIG_copy["keys"].isin(indexes_of_the_group),:]  
      # Create relplot
        g = sns.relplot(
            data=data,
            x="seconds",
            y=column_to_print,
            hue="sensors",
            col="keys",        # separate subplot per key
            kind="line",
            col_wrap=3, 
                height=7,    # default = 5
            aspect=1, # width = height × aspect (so 6 × 1.5 = 9 inches wide per subplot
            palette=column_names_keys_color_values,  # <<< ensures the same colors across all subplots  
            linewidth=2,
           facet_kws={
            "sharex": False,
            "sharey": False       
    
        })
        

        # >>> ADD THIS PART <<<
        for ax in g.axes.flat:
            ax.xaxis.set_major_locator(MultipleLocator(30))
            ax.xaxis.set_minor_locator(MultipleLocator(10))
            ax.grid(True, which='both', linestyle=':', linewidth=0.5)
            
 
        
    # Get the horizontal and  vertical line position for this experiment
        for key_value, ax in g.axes_dict.items():
            for column_name,color in column_names_keys_color_values.items():
                #value to show the time that source is inserted
          
                userInputDataRow = userInputData.loc[key_value,:]
            #    x_position = f"side-right-wall {userInputDataRow['side-right-wall']},side-left-wall {userInputDataRow['side-left-wall']} \n"
            #    y_position = f"front-wall {userInputDataRow['front-wall']},back-wall {userInputDataRow['back-wall']} \n"
                
                euclidian_distances = (
                                      f"distance from Id0 sensor {userInputDataRow['Euclidian distance to Id=0:BME680:breathVocEquivalent']}\n",
                                      f"distance from Id1 sensor {userInputDataRow['Euclidian distance to Id=1:BME680:breathVocEquivalent']}\n",
                                      f"distance from Id2 sensor {userInputDataRow['Euclidian distance to Id=2:BME680:breathVocEquivalent']}\n",
                )
                subtitle=  (
                            f"At experiment with key {key_value}\n datetime:{userInputDataRow['actual timestamp StartingExperiment']}\n", 
                            f"experimentState:{userInputDataRow['experimentState']}\n",
                            f"x-axis: {userInputDataRow['x axis']} , y-axis: {userInputDataRow['y axis']}\n"
                )
                if ("distance"  in type_of_other_grouping):
                   subtitle = subtitle + euclidian_distances  
                ax.set_title(subtitle, fontsize=9)   
            g.fig.suptitle(f"Group: {group_name}", fontsize=16)
        
            g.fig.subplots_adjust(
                    top=0.75,   # space for overall title
                    wspace=0.2, # horizontal space between subplots
                    hspace=0.3 # vertical space between subplots
                   )

        plt.show()   
             

In [ ]:
def plot_all_positions(userInputData):
    room_other_grouping = userInputData.groupby(["x axis","y axis"]).groups
    
    for group_name,indexes_of_the_group in room_other_grouping.items(): 
        sample_row_of_the_group = userInputData.loc[indexes_of_the_group[0],:]
        plot_position(userInputData,sample_row_of_the_group)        

plot_all_positions(userInputData)

room_other_grouping = userInputData.groupby(["room","are-doors-opened","x axis","y axis"]).groups
type_of_other_grouping = ["door-opened","position"]
printDataBasedOnDate("VOC rolling average",userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)

room_other_grouping = userInputData.groupby(["room","are-doors-opened","date of experiment"]).groups
type_of_other_grouping = ["door-opened","date of experiment"]
printDataBasedOnDate("VOC rolling average",userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)

In [ ]:
room_other_grouping = userInputData.groupby(["room","are-doors-opened","x axis","y axis"]).groups
type_of_other_grouping = ["door-opened","position"]
printDataBasedOnDate("VOC original",userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)

In [ ]:
room_other_grouping = userInputData.groupby(["room","are-doors-opened","x axis","y axis"]).groups
type_of_other_grouping = ["door-opened","position"]
printDataBasedOnDate("VOC",userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)

In [ ]:
room_other_grouping = userInputData.groupby(["room","are-doors-opened","date of experiment"]).groups
type_of_other_grouping = ["door-opened","date of experiment"]
printDataBasedOnDate("VOC",userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)

In [ ]:
room_other_grouping = userInputData.groupby(["room","are-doors-opened","x axis","y axis"]).groups
type_of_other_grouping = ["door-opened","position"]
printDataBasedOnDate("VOC rolling average",userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)

In [ ]:
room_other_grouping = userInputData.groupby(["room","are-doors-opened","date of experiment"]).groups
type_of_other_grouping = ["door-opened","date of experiment"]
printDataBasedOnDate("VOC rolling average",userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)

In [ ]:
room_other_grouping = userInputData.groupby(["room","are-doors-opened","x axis","y axis"]).groups
type_of_other_grouping = ["door-opened","position"]
printDataBasedOnDate("VOC-capped",userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)

In [ ]:
room_other_grouping = userInputData.groupby(["room","are-doors-opened","date of experiment"]).groups
type_of_other_grouping = ["door-opened","date of experiment"]
printDataBasedOnDate("VOC-capped",userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)

In [ ]:
room_other_grouping = userInputData.groupby(["room","are-doors-opened","x axis","y axis"]).groups
type_of_other_grouping = ["door-opened","position"]
printDataBasedOnDate("VOC gradient",userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)

In [ ]:
timeSeriesData_BIG

In [ ]:
userInputData

In [ ]:
mask = (timeSeriesData_BIG["seconds"] == 299) & (timeSeriesData_BIG["keys"] == key)
index_of_edge_max = []
column_to_find_values = "VOC rolling average"

keys_grouping = timeSeriesData_BIG.groupby(["keys"]).groups
for key in keys_grouping.keys():
    mask = (timeSeriesData_BIG["seconds"] == 299) & (timeSeriesData_BIG["keys"] == key) 
    data = timeSeriesData_BIG.loc[mask,column_to_find_values]
    column_with_max_value = userInputData.loc[key,f"sensor with MAX value experiment"]
    max_value = userInputData.loc[key,column_with_max_value]
    
    if max_value in data.values:
        print(key)
        print(data)
        print(column_with_max_value)
        print(max_value)
        index_of_edge_max.append(key)

In [ ]:
index_of_edge_max

In [ ]:
from scipy.signal import find_peaks


In [ ]:
column_to_create = "is peak"
timeSeriesData_BIG.loc[:,column_to_create] = np.nan

key_sensor_grouping = timeSeriesData_BIG.groupby(["keys","sensors"]).groups


for key_sensor_key in key_sensor_grouping.keys():
    key = key_sensor_key[0]
    sensor = key_sensor_key[1]
    mask = (timeSeriesData_BIG["keys"] == key) & (timeSeriesData_BIG["sensors"] == sensor) & (timeSeriesData_BIG["seconds"])
    peaks, properties = find_peaks(( (1) * timeSeriesData_BIG.loc[mask,"VOC rolling average"].to_numpy()),plateau_size=1,distance=15)


    # Get the index labels corresponding to the peaks
    subset = timeSeriesData_BIG.loc[mask]
    peak_indices = subset.iloc[peaks].index
    (timeSeriesData_BIG.loc[mask & (timeSeriesData_BIG.index.isin(peak_indices)),column_to_create]) = "positive peak"


    #check if last member is 

for key_sensor_key in key_sensor_grouping.keys():
    key = key_sensor_key[0]
    sensor = key_sensor_key[1]
    mask = (timeSeriesData_BIG["keys"] == key) & (timeSeriesData_BIG["sensors"] == sensor) & (timeSeriesData_BIG["seconds"])
    peaks, properties = find_peaks(( (-1) * timeSeriesData_BIG.loc[mask,"VOC rolling average"].to_numpy()),plateau_size=1,distance=15)


    # Get the index labels corresponding to the peaks
    subset = timeSeriesData_BIG.loc[mask]
    peak_indices = subset.iloc[peaks].index
    (timeSeriesData_BIG.loc[mask & (timeSeriesData_BIG.index.isin(peak_indices)),column_to_create]) = "negative peak"


    #check if last member is 

In [ ]:
column_names_keys_color_values = {"Id=0:BME680:breathVocEquivalent":"blue","Id=1:BME680:breathVocEquivalent":"green","Id=2:BME680:breathVocEquivalent":"yellow"}
markes = {"positive peak": "s", "negative peak": "^"}
column_to_showcase = "is peak"

key_sensor_grouping = timeSeriesData_BIG.groupby(["keys"]).groups
for key in key_sensor_grouping.keys():
    row = userInputData.loc[key]
    mask = ((timeSeriesData_BIG[column_to_showcase]== "positive peak") | (timeSeriesData_BIG[column_to_showcase]== "negative peak")) & (timeSeriesData_BIG["keys"]==key)
    data = timeSeriesData_BIG.loc[mask,:].copy()
    if not data.empty: 
        g = sns.relplot(
            data=data,
            x="seconds",
            y='VOC rolling average-capped',
            hue='sensors',
            kind="scatter",
            style = "is peak",
                            s=120,   # 👈 increase marker size (default ~40)

            markers = markes,
            palette = column_names_keys_color_values,
            height=4,        # height of each facet
            aspect=2,          # width/height ratio → total width = height * aspect
        )
            # >>> ADD THIS PART <<<
        for ax in g.axes.flat:
            ax.set_xlim(-60, 300)
            ax.xaxis.set_major_locator(MultipleLocator(30))
            ax.xaxis.set_minor_locator(MultipleLocator(10))
            ax.grid(True, which='both', linestyle=':', linewidth=0.5)
        
        plt.title(f"Peaks of key {key}")
        plt.show()



    
       # palette=column_names_keys_color_values,  # <<< ensures the same colors across all subplots  

       


In [ ]:
column_to_create = "is peak standard scaler"
column_to_take_from  ="VOC rolling average standard scaled per sensor"
timeSeriesData_BIG.loc[:,column_to_create] = np.nan

key_sensor_grouping = timeSeriesData_BIG.groupby(["keys","sensors"]).groups


for key_sensor_key in key_sensor_grouping.keys():
    key = key_sensor_key[0]
    sensor = key_sensor_key[1]
    mask = (timeSeriesData_BIG["keys"] == key) & (timeSeriesData_BIG["sensors"] == sensor) & (timeSeriesData_BIG["seconds"] >= 0)
    peaks, properties = find_peaks(( (1) * timeSeriesData_BIG.loc[mask,column_to_take_from].to_numpy()),plateau_size=1,distance=15)


    # Get the index labels corresponding to the peaks
    subset = timeSeriesData_BIG.loc[mask]
    peak_indices = subset.iloc[peaks].index
    (timeSeriesData_BIG.loc[mask & (timeSeriesData_BIG.index.isin(peak_indices)),column_to_create]) = "positive peak"


    #check if last member is 

for key_sensor_key in key_sensor_grouping.keys():
    key = key_sensor_key[0]
    sensor = key_sensor_key[1]
    mask = (timeSeriesData_BIG["keys"] == key) & (timeSeriesData_BIG["sensors"] == sensor) & (timeSeriesData_BIG["seconds"] >= 0)
    peaks, properties = find_peaks(( (-1) * timeSeriesData_BIG.loc[mask,column_to_take_from].to_numpy()),plateau_size=1,distance=15)


    # Get the index labels corresponding to the peaks
    subset = timeSeriesData_BIG.loc[mask]
    peak_indices = subset.iloc[peaks].index
    (timeSeriesData_BIG.loc[mask & (timeSeriesData_BIG.index.isin(peak_indices)),column_to_create]) = "negative peak"


    #check if last member is 

In [ ]:
timeSeriesData_BIG

In [ ]:
column_names_keys_color_values = {"Id=0:BME680:breathVocEquivalent":"blue","Id=1:BME680:breathVocEquivalent":"green","Id=2:BME680:breathVocEquivalent":"yellow"}
markes = {"positive peak": "s", "negative peak": "^"}
column_to_showcase =  "is peak standard scaler"
key_sensor_grouping = timeSeriesData_BIG.groupby(["keys"]).groups
for key in key_sensor_grouping.keys():
    row = userInputData.loc[key]
    mask = ((timeSeriesData_BIG[column_to_showcase]== "positive peak") | (timeSeriesData_BIG[column_to_showcase]== "negative peak")) & (timeSeriesData_BIG["keys"]==key)
    data = timeSeriesData_BIG.loc[mask,:].copy()
    if not data.empty: 
        g = sns.relplot(
            data=data,
            x="seconds",
            y='VOC rolling average-capped',
            hue='sensors',
            kind="scatter",
            style = "is peak",
                            s=120,   # 👈 increase marker size (default ~40)

            markers = markes,
            palette = column_names_keys_color_values,
            height=4,        # height of each facet
            aspect=2,          # width/height ratio → total width = height * aspect
        )
            # >>> ADD THIS PART <<<
        for ax in g.axes.flat:
            ax.set_xlim(-60, 300)
            ax.xaxis.set_major_locator(MultipleLocator(30))
            ax.xaxis.set_minor_locator(MultipleLocator(10))
            ax.grid(True, which='both', linestyle=':', linewidth=0.5)
        
        plt.title(f"Peaks of key {key} for column {column_to_showcase}")
        plt.show()



    
       # palette=column_names_keys_color_values,  # <<< ensures the same colors across all subplots  

       


In [ ]:
from scipy.signal import peak_widths
from scipy.signal import peak_prominences

In [ ]:
peak_type = "positive peak"
key_sensor_grouping = timeSeriesData_BIG.groupby(["keys","sensors"]).groups
for key_sensor_key in key_sensor_grouping.keys():
    key = key_sensor_key[0]
    sensor = key_sensor_key[1]
    mask = (timeSeriesData_BIG["keys"] == key) & (timeSeriesData_BIG["sensors"] == sensor) & (timeSeriesData_BIG["seconds"] >= 0)
    is_peak_mask = timeSeriesData_BIG[ "is peak" ] == peak_type
    df_experiment = ((1) * timeSeriesData_BIG.loc[mask,"VOC rolling average"].to_numpy()).copy()
    peaks = timeSeriesData_BIG.loc[mask & is_peak_mask,"seconds"].to_numpy().copy()
    peak_widths_results = peak_widths(df_experiment,peaks)
    peak_prominences_results = peak_prominences(df_experiment,peaks)
    print(f"for {key} and {sensor}")
    for peak_widths_result in peak_widths_results:
        print(f"peak_widths_result:{peak_widths_results}")
    for peak_prominences_result in peak_prominences_results:
        print(f"peak_prominences_result:{peak_prominences_results}")

In [ ]:
peak_type = "negative peak"
key_sensor_grouping = timeSeriesData_BIG.groupby(["keys","sensors"]).groups
for key_sensor_key in key_sensor_grouping.keys():
    key = key_sensor_key[0]
    sensor = key_sensor_key[1]
    mask = (timeSeriesData_BIG["keys"] == key) & (timeSeriesData_BIG["sensors"] == sensor) & (timeSeriesData_BIG["seconds"] >= 0)
    is_peak_mask = timeSeriesData_BIG[ "is peak" ] == peak_type
    df_experiment = ((1) * timeSeriesData_BIG.loc[mask,"VOC rolling average"].to_numpy()).copy()
    peaks = timeSeriesData_BIG.loc[mask & is_peak_mask,"seconds"].to_numpy().copy()
    peak_widths_results = peak_widths(df_experiment,peaks)
    peak_prominences_results = peak_prominences(df_experiment,peaks)
    print(f"for {key} and {sensor}")
    for peak_widths_result in peak_widths_results:
        print(f"peak_widths_result:{peak_widths_results}")
    for peak_prominences_result in peak_prominences_results:
        print(f"peak_prominences_result:{peak_prominences_results}")

In [ ]:
timeSeriesData_BIG

In [ ]:
userInputData

In [ ]:
sensors = timeSeriesData_BIG["sensors"].unique()
# Create a new DataFrame with the same number of rows as df_original
df = pd.DataFrame(
    np.nan,  # fill with NaN
    index=userInputData.index,  # same row index
    columns=sensors  # 3 empty columns
)
column_to_choose = "VOC rolling average gradient"

key_sensor_grouping = timeSeriesData_BIG.groupby(["keys","sensors"]).groups
for key_sensor_key in key_sensor_grouping.keys():
    key = key_sensor_key[0]
    sensor = key_sensor_key[1]
    one_part_of_mask = mask = (timeSeriesData_BIG["keys"] == key) & (timeSeriesData_BIG["sensors"] == sensor)
    before_mean = timeSeriesData_BIG.loc[one_part_of_mask & (timeSeriesData_BIG["seconds"]<0),column_to_choose].mean()
    after_mean = timeSeriesData_BIG.loc[one_part_of_mask & (timeSeriesData_BIG["seconds"]>=0),column_to_choose].mean()
    df.at[key,sensor] = True if after_mean>before_mean else False
print(df.sum())   
df    

In [ ]:
plot_all_positions(userInputData)


In [ ]:
column_to_print = "Id=0:BME680:breathVocEquivalent is gradient mean after biggen than before mean"
condition_to_choose =False
smaller_after_mean= userInputData.loc[userInputData[column_to_print]==condition_to_choose]
type_of_other_grouping =[column_to_print]
room_other_grouping = {column_to_print:smaller_after_mean.index}
printDataBasedOnDate("VOC rolling average-capped",userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)

In [ ]:
euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
group_by_list = ["room","experimentState",*euclidian_distances_columns]
room_other_grouping = userInputData.groupby(group_by_list).groups
type_of_other_grouping = ["experimentState","position","distance"]

sensors = timeSeriesData_BIG["sensors"].unique()
for sensor in sensors:
    mask = timeSeriesData_BIG["sensors"] == sensor
    timeSeriesData_BIG_subset = timeSeriesData_BIG.loc[mask,:]
    print(sensor)
    printDataBasedOnDate("VOC",userInputData,timeSeriesData_BIG_subset,room_other_grouping,type_of_other_grouping)

In [ ]:
euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]

userInputData[euclidian_distances_columns].describe()

In [ ]:
userInputData